In [ ]:
# 1. The Base Case (1D): Car Pooling (LeetCode 1094)
# Problem: Given trips [passengers, start, end] and a vehicle capacity, determine if all trips are possible.
# Concept: 1D Difference Array. Instead of O(N) updates per trip, we record changes at boundaries in O(1) time.
#          - D[start] += val (passengers board)
#          - D[end] -= val (passengers leave instantly at 'end', so no 'end + 1' offset is needed)
# Complexity: Time O(M + max(N)) | Space O(max(N))

class CarPooling:
    def solve(self, trips: list[list[int]], capacity: int) -> bool:
        D = [0] * 1001 
        
        # 1. Boundary Updates
        for numPassengers, start, end in trips:
            D[start] += numPassengers
            D[end] -= numPassengers  
            
        # 2. Prefix Sum Reconstruction
        current_passengers = 0
        for p in D:
            current_passengers += p
            if current_passengers > capacity:
                return False
                
        return True

if __name__ == "__main__":
    print(f"Can complete trips? {CarPooling().solve([[2, 1, 5], [3, 3, 7]], 4)}") # Expected: False

Can complete trips? False


In [ ]:
from sortedcontainers import SortedDict

# 2. Sparse Timelines: My Calendar III (LeetCode 732)
# Problem: Find the maximum overlapping events on a massive timeline (up to 10^9). 
#          Standard arrays will cause Memory Limit Exceeded (MLE).
# Concept: Sweep Line Algorithm. We use a self-sorting dictionary (TreeMap equivalent) 
#          to only store timestamps where a change actually occurs.
# Complexity: Time O(N^2) (for N queries) | Space O(N)

class MyCalendarThree:
    def __init__(self):
        self.timeline = SortedDict()

    def book(self, startTime: int, endTime: int) -> int:
        self.timeline[startTime] = self.timeline.get(startTime, 0) + 1
        self.timeline[endTime] = self.timeline.get(endTime, 0) - 1
        
        current_active = max_overlap = 0
        
        # Sweep only across modified points
        for time in self.timeline:
            current_active += self.timeline[time]
            max_overlap = max(max_overlap, current_active)
            
        return max_overlap

if __name__ == "__main__":
    calendar = MyCalendarThree()
    calendar.book(10, 20)
    calendar.book(50, 60)
    calendar.book(10, 40)
    print(f"Max overlap: {calendar.book(5, 15)}") # Expected: 3

In [ ]:
# 3. Spatial Dimensions (2D): Increment Submatrices (LeetCode 2536)
# Problem: Add 1 to all cells in a 2D matrix from (r1, c1) to (r2, c2) for Q queries.
# Concept: 2D Difference Array (Inclusion-Exclusion). Place 4 boundary markers to define the area in O(1):
#          1. Start wave: +1 at top-left
#          2. Stop vertical bleed: -1 at bottom-left (r2+1)
#          3. Stop horizontal bleed: -1 at top-right (c2+1)
#          4. Re-balance double subtraction: +1 at bottom-right
# Complexity: Time O(Q + N^2) | Space O(N^2)

class IncrementSubmatrices:
    def solve(self, n: int, queries: list[list[int]]) -> list[list[int]]:
        # n+1 padding prevents IndexError on boundaries
        D = [[0] * (n + 1) for _ in range(n + 1)]
        
        # 1. O(1) Boundary Updates
        for r1, c1, r2, c2 in queries:
            D[r1][c1] += 1
            D[r2 + 1][c1] -= 1
            D[r1][c2 + 1] -= 1
            D[r2 + 1][c2 + 1] += 1
            
        # 2. 2D Prefix Sum Reconstruction (In-place)
        for r in range(n):
            for c in range(n):
                if r > 0: D[r][c] += D[r - 1][c]             
                if c > 0: D[r][c] += D[r][c - 1]             
                if r > 0 and c > 0: D[r][c] -= D[r - 1][c - 1] 
                    
        return [row[:n] for row in D[:n]]

if __name__ == "__main__":
    res = IncrementSubmatrices().solve(3, [[1, 1, 2, 2], [0, 0, 1, 1]])
    for row in res: print(row)
    # Expected: [1, 1, 0], [1, 2, 1], [0, 1, 1]